# Подключение LLM

## Задание 1. Первый запрос к модели через `/api/generate`

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434"
LLM_MODEL = "qwen2.5:1.5b"


def generate_answer(prompt: str) -> str:
    """Отправляет промпт в Ollama и возвращает сгенерированный ответ."""

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False,
        },
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()

    return data["response"].strip()


# Пробуем
answer = generate_answer(prompt="Что такое RAG в одном предложении?")
print(answer)


## Задание 2. Построение RAG промпта

In [ ]:
PROMPT_TEMPLATE = """Ты ассистент по документации Ollama.
Отвечай на вопрос пользователя, опираясь только на приведённые ниже фрагменты документации.
Если ответа во фрагментах нет - честно скажи, что не знаешь.
Ответ давай на русском языке.
В конце ответа на отдельной строке укажи источник в формате: Источник: <имя файла>.

Фрагменты документации:
{context}

Вопрос пользователя: {question}

Ответ:"""


def build_prompt(question: str, retrieved: list[dict]) -> str:
    """Собирает промпт из системной инструкции, контекста и вопроса."""
    context_parts = []
    for r in retrieved:
        context_parts.append(f"[файл: {r['source']}]\n{r['text']}")
    context = "\n\n---\n\n".join(context_parts)
    return PROMPT_TEMPLATE.format(context=context, question=question)

In [ ]:
from rag import build_index, vector_search

rag = build_index("docs")

# Тестовый прогон
question = "Как сменить папку, в которой Ollama хранит модели?"

retrieved = vector_search(rag=rag, question=question, top_k=3)

prompt = build_prompt(question, retrieved)

print("Промпт:")
print(prompt[:500], "...\n")

print("Ответ модели:")
answer = generate_answer(prompt)
print(answer)
